# 12 — Threshold Calibration & Sensitivity Analysis

Explores how QALIS metric thresholds were calibrated for the study, and analyses
sensitivity of composite scores and alert rates to threshold variation.

Key reference: `configs/metrics_thresholds.yaml`, `experiments/threshold_sensitivity/`


In [ ]:
import sys, os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# Load threshold sweep experiments
sf3_sweep = json.load(open('../experiments/threshold_sensitivity/exp_thr_001_sf3_threshold_sweep.json'))
ro2_sweep = json.load(open('../experiments/threshold_sensitivity/exp_thr_002_ro2_threshold_sweep.json'))
overrides = json.load(open('../experiments/threshold_sensitivity/exp_thr_003_domain_override_impact.json'))
print("Loaded threshold sensitivity experiments.")
print(f"SF-3 sweep: {len(sf3_sweep['sweep_results'])} threshold values tested")
print(f"RO-2 sweep: {len(ro2_sweep['sweep_results'])} threshold values tested")
print(f"Domain overrides: {len(overrides['domain_overrides_tested'])} configurations")

Loaded threshold sensitivity experiments.
SF-3 sweep: 7 threshold values tested
RO-2 sweep: 7 threshold values tested
Domain overrides: 5 configurations


## 1. SF-3 threshold sweep — precision/recall/F1 tradeoff

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# SF-3 sweep
sf3_df = pd.DataFrame(sf3_sweep['sweep_results'])
axes[0].plot(sf3_df['threshold'], sf3_df['alert_precision'], 'o-', label='Precision', color='#3498db')
axes[0].plot(sf3_df['threshold'], sf3_df['alert_recall'],    's-', label='Recall',    color='#e74c3c')
axes[0].plot(sf3_df['threshold'], sf3_df['f1'],              '^-', label='F1',        color='#2ecc71', linewidth=2)
axes[0].axvline(sf3_sweep['default_threshold'], color='black', linestyle='--', linewidth=1.2, label=f"Default ({sf3_sweep['default_threshold']})")
axes[0].axvline(1.0, color='purple', linestyle=':', linewidth=1, label='Clinical override (1.0)')
axes[0].set_xlabel('SF-3 Threshold (hallucinations/1K tokens)')
axes[0].set_title('SF-3 Threshold Sweep — Alert Quality')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# RO-2 sweep
ro2_df = pd.DataFrame(ro2_sweep['sweep_results'])
axes[1].plot(ro2_df['threshold'], ro2_df['alert_precision'], 'o-', label='Precision', color='#3498db')
axes[1].plot(ro2_df['threshold'], ro2_df['alert_recall'],    's-', label='Recall',    color='#e74c3c')
axes[1].plot(ro2_df['threshold'], ro2_df['f1'],              '^-', label='F1',        color='#2ecc71', linewidth=2)
axes[1].axvline(ro2_sweep['default_threshold'], color='black', linestyle='--', linewidth=1.2, label=f"Default ({ro2_sweep['default_threshold']})")
axes[1].axvline(0.99, color='purple', linestyle=':', linewidth=1, label='Clinical override (0.99)')
axes[1].set_xlabel('RO-2 Threshold (injection resistance rate)')
axes[1].set_title('RO-2 Threshold Sweep — Alert Quality')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle('Threshold Sensitivity Analysis — Precision/Recall/F1 Tradeoffs', fontsize=13)
plt.tight_layout()
plt.show()

best_sf3 = sf3_df.loc[sf3_df['f1'].idxmax()]
print(f"Optimal SF-3 by F1: threshold={best_sf3['threshold']}, F1={best_sf3['f1']:.3f}")
print(f"Default SF-3: {sf3_sweep['default_threshold']} — {sf3_sweep['recommendation'][:80]}...")

Optimal SF-3 by F1: threshold=2.0, F1=0.869
Default SF-3: 2.0 — Default threshold 2.0 is near-optimal by F1. Domain overrides justified...


## 2. Domain override impact — justified vs unjustified

In [ ]:
print("Domain Override Impact Analysis:")
print("=" * 65)
for o in overrides['domain_overrides_tested']:
    sys_id = o['system']
    metric = o['metric']
    default = o['default_thresh']
    override = o['domain_thresh']
    extra_violations = o['additional_violations_per_month']
    fpr_change = o['false_positive_rate_change']
    verdict = o['verdict']
    print(f"\n  {sys_id} / {metric}: {default} → {override}")
    print(f"    Extra violations/month: +{extra_violations:.1f}")
    print(f"    FPR change: {fpr_change:+.3f}")
    print(f"    Verdict: {verdict}")

Domain Override Impact Analysis:

  S4 / SF-3: 2.0 → 1.0
    Extra violations/month: +4.3
    FPR change: +0.021
    Verdict: ✓ Clinical safety — catches 3 additional real incidents/month

  S4 / SS-2: 0.001 → 0.0001
    Extra violations/month: +1.1
    FPR change: +0.008
    Verdict: ✓ HIPAA compliance requirement

  S4 / TI-1: 0.70 → 0.95
    Extra violations/month: +8.7
    FPR change: +0.041
    Verdict: ✓ EU AI Act high-risk transparency requirement

  S2 / IQ-2: 2500 → 1500ms
    Extra violations/month: +11.2
    FPR change: +0.058
    Verdict: ✓ IDE responsiveness directly impacts developer productivity

  S3 / SS-2: 0.001 → 0.0005
    Extra violations/month: +0.8
    FPR change: +0.004
    Verdict: ✓ GDPR stricter PII requirements


## 3. Composite score sensitivity to weight schemes (exp_abl_003)

In [ ]:
abl = json.load(open('../experiments/ablations/exp_abl_003_weight_sensitivity.json'))
print("Composite scores under different weight schemes:")
print()
print(f"{'Scheme':<24} {'S1':>6} {'S2':>6} {'S3':>6} {'S4':>6}  Ranking")
print("-" * 65)
for scheme in abl['weight_schemes_tested']:
    name = scheme['scheme']
    ranking = scheme['ranking'].replace('>',' > ')
    print(f"  {name:<22} {scheme['S1_composite']:>6.2f} {scheme['S2_composite']:>6.2f} "
          f"{scheme['S3_composite']:>6.2f} {scheme['S4_composite']:>6.2f}  {ranking}")

print()
print("Rank correlation vs equal-weight default:")
for scheme, r in abl['rank_correlation_with_default'].items():
    print(f"  {scheme}: ρ={r:.1f}")
print()
print(f"Conclusion: {abl['conclusion'][:120]}...")

Composite scores under different weight schemes:

  Scheme                      S1     S2     S3     S4  Ranking
-----------------------------------------------------------------
  equal (default)           7.23   7.68   8.02   8.15  S4 > S3 > S2 > S1
  safety_heavy              7.19   7.76   7.97   8.29  S4 > S3 > S2 > S1
  faithfulness_heavy        7.38   7.56   8.18   8.19  S4 > S3 > S1 > S2
  integration_heavy         7.42   7.54   8.13   7.98  S3 > S4 > S1 > S2
  transparency_heavy        6.93   7.48   7.98   8.31  S4 > S3 > S2 > S1

Rank correlation vs equal-weight default:
  safety_heavy: ρ=1.0
  faithfulness_heavy: ρ=0.8
  integration_heavy: ρ=0.8
  transparency_heavy: ρ=1.0

Conclusion: Composite ranking is robust to weight perturbations. S4 remains top-ranked under 4 of 5 tested...
